# 1. Create a database connection 🔌🏦

In [27]:
import pandas as pd
from sqlalchemy import create_engine, types
from sqlalchemy import text # to be able to pass string

In [28]:
# 1. Set up connection (update credentials and host!)
from dotenv import dotenv_values

config = dotenv_values() 

# define variables for the login
pg_user = config['POSTGRES_USER']  # align the key label with your .env file !
pg_host = config['POSTGRES_HOST']
pg_port = config['POSTGRES_PORT']
pg_db = config['POSTGRES_DB']
pg_schema = config['POSTGRES_SCHEMA']
pg_pass = config['POSTGRES_PASS']

url = f'postgresql://{pg_user}:{pg_pass}@{pg_host}:{pg_port}/{pg_db}'

engine = create_engine(url, echo=False)



In [3]:
my_schema = "team_3"


mart_customers = pd.read_sql(f"SELECT * FROM {my_schema}.mart_customers;", engine)
mart_discount_coupon = pd.read_sql(f"SELECT * FROM {my_schema}.mart_discount_coupon;", engine)
mart_holidays_2019_us = pd.read_sql(f"SELECT * FROM {my_schema}.mart_holidays_2019_us;", engine)
mart_marketing_spend = pd.read_sql(f"SELECT * FROM {my_schema}.mart_marketing_spend;", engine)
mart_online_sales = pd.read_sql(f"SELECT * FROM {my_schema}.mart_online_sales;", engine)
mart_tax_amount = pd.read_sql(f"SELECT * FROM {my_schema}.mart_tax_amount;", engine)
mart_all_data = pd.read_sql(f"SELECT * FROM {my_schema}.mart_all_data;", engine)


# Step 1: Create Month String Columns


In [4]:
# Ensure transaction_date is datetime
mart_all_data['transaction_date'] = pd.to_datetime(mart_all_data['transaction_date'])
mart_all_data['transaction_month_str'] = mart_all_data['transaction_date'].dt.strftime('%Y-%m')

# Ensure marketing spend date is datetime and build month string
mart_marketing_spend['date'] = pd.to_datetime(mart_marketing_spend['date'])
mart_marketing_spend['month_str'] = mart_marketing_spend['date'].dt.strftime('%Y-%m')

# Calculate total marketing spend per day
mart_marketing_spend['total_spend'] = mart_marketing_spend['offline_spend'].fillna(0) + mart_marketing_spend['online_spend'].fillna(0)


# Step 2: Calculate Monthly Metrics
Revenue, Orders, New Customers, Spend

In [5]:
# Monthly revenue
monthly_revenue = mart_all_data.groupby('transaction_month_str')['revenue'].sum().reset_index(name='total_revenue')

# Monthly orders
monthly_orders = mart_all_data.groupby('transaction_month_str')['transaction_id'].nunique().reset_index(name='order_count')

# Monthly marketing spend
monthly_spend = mart_marketing_spend.groupby('month_str')['total_spend'].sum().reset_index()

# Monthly new customers (first purchase in month)
mart_all_data['customer_first_month'] = mart_all_data.groupby('customer_id')['transaction_date'].transform('min').dt.strftime('%Y-%m')
new_customers = mart_all_data.drop_duplicates('customer_id').groupby('customer_first_month')['customer_id'].count().reset_index(name='new_customers')


# Step 3: Merge All Monthly Metrics



In [7]:
summary = monthly_revenue.merge(monthly_spend, left_on='transaction_month_str', right_on='month_str', how='outer')
summary = summary.merge(monthly_orders, on='transaction_month_str', how='outer')
summary = summary.merge(new_customers, left_on='transaction_month_str', right_on='customer_first_month', how='left')

# KPIs
summary['AOV'] = summary['total_revenue'] / summary['order_count']
summary['CAC'] = summary['total_spend'] / summary['new_customers']
summary['ROAS'] = summary['total_revenue'] / summary['total_spend']

# Step 4: Overall (Lifetime) Metrics


Repeat Purchase Rate

In [8]:
repeat_customers = mart_all_data.groupby('customer_id')['transaction_id'].nunique().reset_index()
repeat_customers['is_repeat'] = repeat_customers['transaction_id'] > 1
repeat_rate = repeat_customers['is_repeat'].mean()

Churn Rate

In [20]:
# Find last purchase date per customer
last_purchase = mart_all_data.groupby('customer_id')['transaction_date'].max().reset_index()
last_purchase['transaction_date'] = pd.to_datetime(last_purchase['transaction_date'])

reference_date = pd.to_datetime('2019-12-31')

# Calculate months since last purchase
last_purchase['months_since'] = ((reference_date - last_purchase['transaction_date']) / pd.Timedelta(days=30)).astype(int)

# Customers with >6 months inactivity are considered churned
churned = last_purchase[last_purchase['months_since'] > 6]

# Churn rate = churned customers / total unique customers
churn_rate = len(churned) / mart_all_data['customer_id'].nunique()


LTV (Average revenue per customer)


In [10]:
ltv = mart_all_data.groupby('customer_id')['revenue'].sum().mean()

LTV:CAC Ratio

In [12]:
avg_cac = summary['CAC'].mean()
ltv_cac_ratio = ltv / avg_cac

Retention Rate

(Percent of customers who purchased in at least 2 different months)

In [16]:
customer_months = mart_all_data.groupby('customer_id')['transaction_month_str'].nunique().reset_index()
customer_months['retained'] = customer_months['transaction_month_str'] > 1
retention_rate = customer_months['retained'].mean()

Retention Rate 6 m

In [14]:
mart_all_data['transaction_date'] = pd.to_datetime(mart_all_data['transaction_date'])
mart_all_data['first_purchase_month'] = mart_all_data.groupby('customer_id')['transaction_date'].transform('min').dt.to_period('M')
mart_all_data['order_month'] = mart_all_data['transaction_date'].dt.to_period('M')
mart_all_data['months_since_signup'] = (mart_all_data['order_month'] - mart_all_data['first_purchase_month']).apply(lambda x: x.n)
# Customers who purchased in months 1–6 (excluding 0, the signup month)
retained_customers = mart_all_data[
    (mart_all_data['months_since_signup'] > 0) & (mart_all_data['months_since_signup'] < 6)
]['customer_id'].unique()

# All unique customers (signups)
all_customers = mart_all_data['customer_id'].unique()

# 6-month retention rate
retention_6m = len(retained_customers) / len(all_customers)
print(f"6-month retention rate: {retention_6m:.2%}")

6-month retention rate: 33.72%


In [21]:
overall_kpis = pd.DataFrame({
    'LTV': [ltv],
    'Repeat_Purchase_Rate': [repeat_rate],
    'Retention_Rate': [retention_rate],
    'Retention_Rate_6m': [retention_6m],
    'Churn_Rate': [churn_rate],
    'LTV:CAC_Ratio': [ltv_cac_ratio]
})

print(overall_kpis)

           LTV  Repeat_Purchase_Rate  Retention_Rate  Retention_Rate_6m  \
0  3560.919407               0.91485        0.395095           0.337193   

   Churn_Rate  LTV:CAC_Ratio  
0    0.267711       2.688159  


# Adding to AWS cloud

In [26]:
summary.to_sql('monthly_kpi_summary', engine, schema=my_schema, if_exists='replace', index=False)
overall_kpis.to_sql('overall_kpis', engine, schema=my_schema, if_exists='replace', index=False)

1

# Download in .csv files

In [ ]:
summary.to_csv('monthly_kpi_summary.csv', index=False)
overall_kpis.to_csv('overall_kpis.csv', index=False)